In [ ]:
import mlflow
import pandas as pd
import numpy as np
MLFLOW_TRACKING_URI = 'http://localhost:5001'
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
summary = {}

In [ ]:
cae = mlflow.search_runs(experiment_ids=16)
cae = pd.merge(cae.groupby(by='tags.mlflow.parentRunId').agg(['mean', 'std']), cae, 
         left_on='tags.mlflow.parentRunId', right_on='run_id')
cae['model'] = 'cae'
summary['cae'] = (cae.filter(regex=r"\('metrics\.(roc-auc).+(mean|std)'\)|params.data_source|params.pipeline|model"))
summary['cae']

In [ ]:
bae = mlflow.search_runs(experiment_ids=11)
bae = pd.merge(bae.groupby(by='tags.mlflow.parentRunId').agg(['mean', 'std']), bae, 
         left_on='tags.mlflow.parentRunId', right_on='run_id')
bae['model'] = 'bae'
summary['bae'] = (bae.filter(regex=r"\('metrics\.(roc-auc).+(mean|std)'\)|params.data_source|params.pipeline|model"))
summary['bae']

In [ ]:
vae = mlflow.search_runs(experiment_ids=17)
vae = pd.merge(vae.groupby(by='tags.mlflow.parentRunId').agg(['mean', 'std']), vae, 
         left_on='tags.mlflow.parentRunId', right_on='run_id')
vae['model'] = 'vae'
summary['vae'] = (vae.filter(regex=r"\('metrics\.(roc-auc).+(mean|std)'\)|params.data_source|params.pipeline|model"))
summary['vae']

In [ ]:
dcgan = mlflow.search_runs(experiment_ids=13)
grouped = dcgan.groupby(by='tags.mlflow.parentRunId').agg(['mean', 'std'])
dcgan = pd.merge(grouped, dcgan, left_on='tags.mlflow.parentRunId', right_on='run_id')
dcgan['model'] = 'dcgan'
summary['dcgan'] = (dcgan.filter(regex=r"\('metrics\.(roc-auc).+(mean|std)'\)|params.data_source|params.pipeline|model"))
summary['dcgan']

In [ ]:
bigan = mlflow.search_runs(experiment_ids=14)
bigan = pd.merge(bigan.groupby(by='tags.mlflow.parentRunId').agg(['mean', 'std']), bigan, 
         left_on='tags.mlflow.parentRunId', right_on='run_id')
bigan['model'] = 'bigan'
summary['bigan'] = (bigan.filter(regex=r"\('metrics\.(roc-auc).+(mean|std)'\)|params.data_source|params.pipeline|model"))
summary['bigan']

In [ ]:
alphagan = mlflow.search_runs(experiment_ids=15)
alphagan = pd.merge(alphagan.groupby(by='tags.mlflow.parentRunId').agg(['mean', 'std']), alphagan, 
         left_on='tags.mlflow.parentRunId', right_on='run_id')
alphagan['model'] = 'alphagan'
summary['alphagan'] = (alphagan.filter(regex=r"\('metrics\.(roc-auc).+(mean|std)'\)|params.data_source|params.pipeline|model"))
summary['alphagan']

In [ ]:
summary_df = pd.concat(summary.values())

In [ ]:
summary_df.to_csv('summary.csv')

In [ ]:
summary_df['params.pipeline'] = summary_df['params.pipeline'].apply(eval).apply(lambda par: par['hist_equalisation'])

In [ ]:
for col_mean, col_std in zip(summary_df.filter(regex=r"\('metrics\.(roc-auc).+(mean)'\)"),
                             summary_df.filter(regex=r"\('metrics\.(roc-auc).+(std)'\)")):
    rounding = lambda x: ('%.3f' % x)[1:]
    summary_df[col_mean[0]] = summary_df[col_mean].apply(rounding) + r' ± ' + summary_df[col_std].apply(rounding)

In [ ]:
summary_df.drop(summary_df.filter(regex=r"\('metrics\.(roc-auc).+(mean|std)'").columns, axis=1, inplace=True)

In [ ]:
df = summary_df.pivot_table(columns=['model', 'params.data_source', 'params.pipeline'], aggfunc='first')
df = df[~(df == r'an ± an')]
df = df.unstack(level=(2, 3)).reorder_levels((1, 0)).sort_index().sort_index(axis=1)


In [ ]:
df[df.isna()] = '-'
df = df.reindex(['cae', 'vae', 'dcgan', 'bigan', 'alphagan'], level=0)
df = df.rename(index={'metrics.roc-auc_mse':'MSE', 
                 'metrics.roc-auc_mse_top_k': 'MSE (top-200)',
                 'metrics.roc-auc_kld': 'KLD',
                 'metrics.roc-auc_l1': 'L1', 
                 'metrics.roc-auc_l1_kld': 'L1 + KLD',
                 'metrics.roc-auc_l1_top_k': 'L1 (top-200)',
                 'metrics.roc-auc_mse_kld': 'MSE + KLD',
                 'metrics.roc-auc_proba': 'Discriminator (D)', 
                 'metrics.roc-auc_coproba': 'Code-Discriminator (C)',
                 'metrics.roc-auc_proba_coproba': 'C + D'}, level=1)
df = df.rename(columns={'XR_HAND':'raw', 
                        'XR_HAND_CROPPED': 'crop',
                        'XR_HAND_PHOTOSHOP': 'full'}, level=0)
df

In [ ]:
df_num = (df.applymap(lambda x: x[0:4]).applymap(lambda x: x if x!='-' else None).astype(float) - 0.50)

def highlight_max(s):
    '''
    highlight the maximum in a Series yellow.
    '''
    is_max = s == s.max()
    return ['background-color: yellow' if v else '' for v in is_max]

df_num.style.apply(highlight_max, axis=1)

In [ ]:
print(df.to_latex(bold_rows=True))